# Rumi on a Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asterisk-labs/rumi/blob/0.27/examples/rumi-gpu.ipynb)

This notebook writes a small Rumi image, reads a window directly into a CUDA tensor, and checks the result against the CPU reader. Select a GPU runtime before running it.

In [ ]:
%pip install -q --pre "geozl==0.20.0a1" "rumi-eo==0.27.0a1"

In [ ]:
from pathlib import Path

import geozl
import numpy as np
import rumi
import torch

assert torch.cuda.is_available(), "Select a GPU runtime in Colab"
print("Rumi", rumi.__version__)
print("GeoZL", geozl.__version__)
print("GPU", torch.cuda.get_device_name(0))

## Write a small image

The PivCo recipe below is supported by the CUDA decoder.

In [ ]:
rng = np.random.default_rng(42)
data = rng.integers(0, 10_000, (4, 512, 512), dtype=np.uint16)

frames = rumi.frames(
    data,
    "b (row h) (col w) -> row col (b h w)",
    tile_size=128,
)
for frame in frames:
    graph = geozl.graph(
        frame.data, "planar>zigzag>transpose>pivco"
    )
    frame.compressed = geozl.compress(frame.data, graph=graph)

path, header = rumi.write(
    Path("scene.rumi"),
    frames,
    bands=["B2", "B3", "B4", "B8"],
    time=["2026-10-31"],
)
print(path, path.stat().st_size, "bytes")

## Read directly into CUDA

Rumi parses the header on the CPU, then writes the selected samples into a PyTorch tensor on `cuda:0`. The call to `cpu()` below is only for the correctness check.

In [ ]:
selection = dict(bands=[2, 3], window=(96, 80, 192, 224))
cpu = rumi.read(path, header, **selection)
gpu = rumi.read(
    path,
    header,
    **selection,
    framework="torch",
    device="cuda:0",
)

assert gpu.is_cuda
np.testing.assert_array_equal(gpu.cpu().numpy(), cpu)
print(gpu.shape, gpu.dtype, gpu.device)

## Batch windows

`read_many` keeps the batch on the same device.

In [ ]:
windows = [
    (0, 0, 128, 128),
    (128, 128, 128, 128),
]
batch = rumi.read_many(
    [path, path],
    [header, header],
    windows=windows,
    framework="torch",
    device="cuda:0",
)

expected = np.stack([
    data[:, row:row + height, col:col + width]
    for row, col, height, width in windows
])
assert batch.is_cuda
np.testing.assert_array_equal(batch.cpu().numpy(), expected)
print(batch.shape, batch.dtype, batch.device)